In [1]:
import sys, tempfile, zipfile, os
sys.path.append('..') 

from scripts.constants import *
import ee, xee, geemap, eemont
import numpy as np
import rioxarray as rxr
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project=GEE_PROJECT_NAME, opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
# Read the raster using rioxarray
raster_path = 'path_to_your_raster.tif'
raster = rxr.open_rasterio(raster_path, masked=True).squeeze()

# Convert the raster to an ee.Image instance
raster_ee = ee.Image(raster_path)

# Print the raster information
print(raster)
print(raster_ee.getInfo())

In [ ]:
# Define the input layer (replace 'your_layer' with the actual layer)
input_layer = ee.Image('your_layer')

# Define the dependent variable (e.g., 'NDVI')
dependent_variable = 'NDVI'

# Define the independent variables (e.g., 'B1', 'B2', 'B3')
independent_variables = ['B1', 'B2', 'B3']

# Sample the input layer to get the training data
training_data = input_layer.select(independent_variables + [dependent_variable]).sample(
    region=your_region,  # Define your region of interest
    scale=30,  # Define the scale in meters
    numPixels=5000  # Define the number of pixels to sample
)

# Convert the training data to a feature collection
training_fc = training_data.randomColumn()

# Split the data into training and testing sets
training_set = training_fc.filter(ee.Filter.lt('random', 0.7))
testing_set = training_fc.filter(ee.Filter.gte('random', 0.7))

# Define the regression model
regression_model = ee.Regression.linear(independent_variables)

# Train the model
trained_model = regression_model.train(training_set, dependent_variable, independent_variables)

# Apply the model to the input layer
predicted = input_layer.select(independent_variables).regression(trained_model)

# Print the model coefficients
print('Model coefficients:', trained_model.coefficients().getInfo())

# Evaluate the model using the testing set
evaluation = trained_model.evaluate(testing_set)
print('Model evaluation:', evaluation.getInfo())

# Display the predicted image
geemap.Map.addLayer(predicted, {}, 'Predicted')
geemap.Map.centerObject(your_region)
geemap.Map